# Explorative Datenanalyse (EDA)

In [ ]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.model_selection import train_test_split

from notebooks.edahelper import fill_spectral_nans

from math import ceil

# Loading Data

In [ ]:
#place test and training data in the data/raw folder
df_train = pd.read_csv(Path.cwd() / "data" / "raw" / "train.csv", index_col=0)
df_test = pd.read_csv(Path.cwd() / "data" / "raw" / "test.csv", index_col=0)

# NaN values
## Empty bands
-> dropping completely 

In [ ]:
empty_cols = df_train.columns[df_train.isna().all()] #.tolist()
df_train = df_train.drop(columns=empty_cols)
df_test = df_test.drop(columns=empty_cols)

print("Dropped columns:\n", empty_cols)


## Single Missing values

In [ ]:
df_train[df_train.isnull().any(axis=1)][df_train.columns[df_train.isna().any()]]

In [ ]:
df_test[df_test.isnull().any(axis=1)][df_test.columns[df_test.isna().any()]]

In [ ]:
nan_columns = df_train.columns[df_train.isna().any()]
print(nan_columns)

In [ ]:
# [link to line diagram] ->  Filling with values from neighboring columns better then with mean of the column (only as backup)
df_train = fill_spectral_nans(df_train, 15, 45)
df_test = fill_spectral_nans(df_test, 15, 45)

## Duplicate

In [ ]:
#Absolute Duplikate
df_train[df_train.drop('id', axis=1).duplicated(keep=False)]

In [ ]:
# Drop duplicates based on all columns except 'id'
df_train.drop_duplicates(subset=df_train.columns.difference(['id']), inplace=True)

In [ ]:

df_train[df_train['X2315'].duplicated(False)]

# Vergleich Test und Trainingsdaten

In [ ]:
df_compare = pd.concat(
    [df_train.assign(type="train"), df_test.assign(type="test")],
    ignore_index=True,
)

In [ ]:
comparison = (
    df_compare.melt(
        id_vars="type",
        value_vars=["AEZ", "Month"],
        var_name="feature",
        value_name="category",
    )
    .groupby(["feature", "category", "type"], dropna=False)
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    comparison,
    x="category",
    y="count",
    color="type",
    barmode="group",
    facet_col="feature",
    labels={"category": "Category", "count": "Number of samples", "type": "Dataset"},
    title="Comparison of AEZ and Month distributions between train and test datasets (absolute)"
)
fig.show()

In [ ]:
comparison = (
    df_compare.melt(
        id_vars="type",
        value_vars=["AEZ", "Month"],
        var_name="feature",
        value_name="category",
    )
    .groupby(["feature", "category", "type"], dropna=False)
    .size()
    .reset_index(name="count")
)

comparison["relative_share"] = comparison["count"] / comparison["type"].map(
    df_compare["type"].value_counts()
) * 100

fig = px.bar(
    comparison,
    x="category",
    y="relative_share",
    color="type",
    barmode="group",
    facet_col="feature",
    labels={
        "category": "Category",
        "relative_share": "Share of dataset (%)",
        "type": "Dataset",
    },
    title="Comparison of AEZ and Month distributions between train and test datasets (relative)"
)
fig.show()

In [ ]:


bands = sorted(
    (
        column
        for column in set(df_train.columns) & set(df_test.columns)
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)

if not bands:
    raise ValueError("No shared spectral bands found between train and test.")

wavelengths_nm = [int(band[1:]) for band in bands]

fig = go.Figure()

for label, df, color in [
    ("Train", df_train, "blue"),
    ("Test", df_test, "orange"),
]:
    fig.add_trace(
        go.Box(
            x=wavelengths_nm * len(df),
            y=df[bands].to_numpy().ravel(),
            name=label,
            offsetgroup=label,
            marker_color=color,
            boxpoints=False,
            hovertemplate=(
                f"{label}<br>Wavelength=%{{x}} nm"
                "<br>Reflectance=%{y}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Reflectance distribution by wavelength: train vs. test",
    xaxis={"title": "Wavelength [nm]", "type": "linear"},
    yaxis_title="Reflectance [%]",
    boxmode="group",
)
fig.show()

In [ ]:
import plotly.graph_objects as go

bands = sorted(
    (
        column
        for column in set(df_train.columns) & set(df_test.columns)
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)

if not bands:
    raise ValueError("No shared spectral bands found between train and test.")

wavelengths_nm = [int(band[1:]) for band in bands]
train_mean = df_train[bands].mean()
test_mean = df_test[bands].mean()

fig = go.Figure()

for label, mean_spectrum, color in [
    ("Train mean", train_mean, "blue"),
    ("Test mean", test_mean, "orange"),
]:
    x_values = []
    y_values = []

    for index, (wavelength, band) in enumerate(zip(wavelengths_nm, bands)):
        if index > 0 and wavelength - wavelengths_nm[index - 1] > 50:
            x_values.append(None)
            y_values.append(None)

        x_values.append(wavelength)
        y_values.append(mean_spectrum[band])

    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            mode="lines",
            name=label,
            line={"color": color},
            hovertemplate="Wavelength=%{x} nm<br>Mean reflectance=%{y}<extra></extra>",
        )
    )

fig.update_layout(
    title="Mean spectrum: train vs. test",
    xaxis_title="Wavelength [nm]",
    yaxis_title="Mean reflectance [%]",
)
fig.show()

In [ ]:


bands = sorted(
    (
        column
        for column in set(df_train.columns) & set(df_test.columns)
        if column.startswith("X") and column[1:].isdigit()
    ),
    key=lambda column: int(column[1:]),
)

if not bands:
    raise ValueError("No shared spectral bands found between train and test.")

wavelengths_nm = [int(band[1:]) for band in bands]

fig = go.Figure()

for label, df, color in [
    ("Train", df_train, "blue"),
    ("Test", df_test, "orange"),
]:
    fig.add_trace(
        go.Box(
            x=wavelengths_nm * len(df),
            y=df[bands].to_numpy().ravel(),
            name=label,
            offsetgroup=label,
            marker_color=color,
            boxpoints=False,
            hovertemplate=(
                f"{label}<br>Wavelength=%{{x}} nm"
                "<br>Reflectance=%{y}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Reflectance distribution by wavelength: train vs. test",
    xaxis={"title": "Wavelength [nm]", "type": "linear"},
    yaxis_title="Reflectance [%]",
    boxmode="group",
)
fig.show()

# Similarities in test dataset

In [ ]:
bands = sorted((column for column in df_train.columns if column.startswith("X") and column[1:].isdigit()),
    key=lambda column: int(column[1:]),
    )

spectra = df_train[bands].to_numpy(dtype=float)

if not np.isfinite(spectra).all():
    raise ValueError("Handle missing or infinite frequency values first.")

sample_ids = (
    df_train["id"].to_numpy()
    if "id" in df_train.columns
    else df_train.index.to_numpy()
)
crops = df_train["Crop"].to_numpy()
stages = df_train["Stage"].to_numpy()

pair_i, pair_j = np.triu_indices(len(df_train), k=1)


def make_pair_table(values, metric_name):
    """Format the upper triangle of a pairwise metric matrix as sample pairs."""
    scores = values[pair_i, pair_j]
    valid = np.isfinite(scores)

    i = pair_i[valid]
    j = pair_j[valid]

    return pd.DataFrame(
        {
            "id x": sample_ids[i],
            "id y": sample_ids[j],
            metric_name: scores[valid],
            "crop x": crops[i],
            "crop y": crops[j],
            "stage x": stages[i],
            "stage y": stages[j],
        }
    )

## Pearson correlation — higher means more similar:

In [ ]:
centered = spectra - spectra.mean(axis=1, keepdims=True)
norms = np.linalg.norm(centered, axis=1)

if np.any(norms == 0):
    raise ValueError("Pearson correlation is undefined for a constant spectrum.")

normalized = centered / norms[:, None]
pearson_matrix = normalized @ normalized.T
pearson_matrix = np.clip(pearson_matrix, -1.0, 1.0)

pearson_pairs = make_pair_table(pearson_matrix, "pearson correlation")
display(pearson_pairs)

## Spectral Angle Mapper (SAM) — angle in degrees; lower means more similar:



In [ ]:
norms = np.linalg.norm(spectra, axis=1)

if np.any(norms == 0):
    raise ValueError("SAM is undefined for a zero spectrum.")

normalized = spectra / norms[:, None]
cosine_matrix = np.clip(normalized @ normalized.T, -1.0, 1.0)
sam_matrix = np.degrees(np.arccos(cosine_matrix))

sam_pairs = make_pair_table(sam_matrix, "SAM angle (degrees)")
display(sam_pairs)

## Euclidean distance — lower means more similar; sensitive to reflectance scale:

In [ ]:
squared_norms = np.sum(spectra**2, axis=1)
squared_distances = (
    squared_norms[:, None] + squared_norms[None, :] - 2 * (spectra @ spectra.T)
)
euclidean_matrix = np.sqrt(np.maximum(squared_distances, 0.0))

euclidean_pairs = make_pair_table(euclidean_matrix, "euclidean distance")
display(euclidean_pairs)

## Spectral Information Divergence (SID) — lower means more similar; requires nonnegative spectra:

In [ ]:
if np.any(spectra < 0):
    raise ValueError("SID requires nonnegative spectral values.")

# Convert each spectrum to a probability distribution across its bands.
positive = np.maximum(spectra, 1e-12)
probabilities = positive / positive.sum(axis=1, keepdims=True)
log_probabilities = np.log(probabilities)

cross_terms = probabilities @ log_probabilities.T
self_terms = np.sum(probabilities * log_probabilities, axis=1)
sid_matrix = (
    self_terms[:, None]
    + self_terms[None, :]
    - cross_terms
    - cross_terms.T
)
sid_matrix = np.maximum(sid_matrix, 0.0)

sid_pairs = make_pair_table(sid_matrix, "SID")
display(sid_pairs)

## Cosine similarity 

In [ ]:
norms = np.linalg.norm(spectra, axis=1)

if np.any(norms == 0):
    raise ValueError("Cosine similarity is undefined for a zero spectrum.")

normalized = spectra / norms[:, None]
cosine_matrix = np.clip(normalized @ normalized.T, -1.0, 1.0)

cosine_pairs = make_pair_table(cosine_matrix, "cosine similarity")
display(cosine_pairs)

In [ ]:
cosine_pairs[(cosine_pairs["cosine similarity"] > 0.99) & (cosine_pairs["crop x"] != cosine_pairs["crop y"]) & (cosine_pairs["stage x"] == cosine_pairs["stage y"])].sort_values(by="cosine similarity", ascending=False)

In [ ]:
cosine_pairs[(cosine_pairs["cosine similarity"] > 0.99) & (cosine_pairs["stage x"] != cosine_pairs["stage y"]) & (cosine_pairs["crop x"] == cosine_pairs["crop y"])].sort_values(by="cosine similarity", ascending=False)

In [ ]:
cosine_pairs[(cosine_pairs["cosine similarity"] > 0.99) & ((cosine_pairs["stage x"] != cosine_pairs["stage y"]) | (cosine_pairs["crop x"] != cosine_pairs["crop y"]))].sort_values(by="cosine similarity", ascending=False)

# Klassenverteilungen und Statifizierung der Daten
## Verteilungen Crop, Stage

In [ ]:
crop_stage_counts = pd.crosstab(df_train["Crop"], df_train["Stage"])

fig = px.imshow(
    crop_stage_counts,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Stage", "y": "Crop", "color": "Anzahl"},
    title="Anzahl der Trainingsproben je Crop und Stage",
)
fig.show()

In [ ]:
# AEZ × Crop
aez_crop = pd.crosstab(df_train["AEZ"], df_train["Crop"]).sort_index()

fig = px.imshow(
    aez_crop,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Crop", "y": "AEZ", "color": "Anzahl"},
    title="Trainingsproben je AEZ und Crop",
)
fig.show()

# Month × Stage, one panel per crop
crops = sorted(df_train["Crop"].dropna().unique())
months = sorted(df_train["Month"].dropna().unique())
stages = sorted(df_train["Stage"].dropna().unique())

n_cols = 3
n_rows = ceil(len(crops) / n_cols)
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=crops,
    shared_xaxes=True,
    shared_yaxes=True,
)

for i, crop in enumerate(crops):
    row, col = divmod(i, n_cols)
    counts = pd.crosstab(
        df_train.loc[df_train["Crop"] == crop, "Month"],
        df_train.loc[df_train["Crop"] == crop, "Stage"],
    ).reindex(index=months, columns=stages, fill_value=0)

    fig.add_trace(
        go.Heatmap(
            z=counts.values,
            x=stages,
            y=months,
            text=counts.values,
            texttemplate="%{text}",
            coloraxis="coloraxis",
        ),
        row=row + 1,
        col=col + 1,
    )

fig.update_layout(
    title="Trainingsproben je Month und Stage, aufgeteilt nach Crop",
    coloraxis={"colorscale": "Blues"},
)
fig.update_xaxes(
    title_text="Stage",
    tickmode="array",
    tickvals=stages,
    ticktext=stages,
    tickangle=45,
    row=n_rows,
)
fig.update_yaxes(
    title_text="Month",
    tickmode="array",
    tickvals=months,
    ticktext=[str(month) for month in months],
)
fig.show()

In [ ]:
crop_stage_counts = (
    df_train.groupby(["Crop", "Stage"], observed=True)
    .size()
    .rename("absolute_count")
    .to_frame()
)
crop_stage_counts["relative_count"] = (
    crop_stage_counts["absolute_count"] / len(df_train)
)

crop_stage_counts.round(2)

In [ ]:

SEED = 42


crop_stage = pd.MultiIndex.from_frame(df_train[["Crop", "Stage"]])
strata = pd.factorize(crop_stage)[0]

df_train_split, df_validation = train_test_split(
    df_train,
    test_size=0.2,
    stratify=strata,
)

In [ ]:
crop_stage_counts = (
    df_validation.groupby(["Crop", "Stage"], observed=True)
    .size()
    .rename("absolute_count")
    .to_frame()
)
crop_stage_counts["relative_count"] = (
    crop_stage_counts["absolute_count"] / len(df_validation)
)

crop_stage_counts.round(2)

# Bänder
Im Datensatz
437 - 923 <br>
983 - 1094 br>
1175 - 1316 <br>
1518 - 1760 <br>
2063 - 2345 <br>


Google:
400 – 450 nm | Coastal Aerosol / Deep Blue <br>
450 – 510 nm | Blue <br>
500 – 600 nm | Green<br>
600 – 700 nm | Red<br>
700 – 800 nm | Red Edge<br>
800 – 1,000 nm | Near-Infrared (NIR)<br>
1,300 – 1,400 nm | Cirrus 
1,550 – 1,750 nm | SWIR 1<br>
2,100 – 2,300 nm | SWIR 2<br>





In [ ]:
from math import ceil

import plotly.graph_objects as go
from plotly.subplots import make_subplots

BANDS = [
    column for column in df_train.columns
    if column.startswith("X") and column[1:].isdigit()
]
WAVELENGTHS = [int(column[1:]) for column in BANDS]
GROUPS = list(df_train.groupby(["Crop", "Stage"], sort=True, observed=True))

N_COLS = 2  # Set to 1, 2, or 3.
if N_COLS not in (1, 2, 3):
    raise ValueError("N_COLS must be 1, 2, or 3")


def make_spectral_figure(title):
    """Create a subplot grid with one panel per observed crop-stage pair."""
    n_cols = min(N_COLS, len(GROUPS))
    n_rows = ceil(len(GROUPS) / n_cols)
    subplot_titles = [
        f"{crop} — {stage} (n={len(group)})"
        for (crop, stage), group in GROUPS
    ]
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        shared_xaxes=True,
        shared_yaxes=True,
    )
    fig.update_layout(
        title=title,
        height=max(350, 300 * n_rows),
        width=500 * n_cols,
        showlegend=False,
        hovermode="closest",
    )
    return fig, n_cols

In [ ]:
fig, n_cols = make_spectral_figure(
    "Reflectance distribution by wavelength, crop, and stage"
)

ordered_bands = sorted(BANDS, key=lambda band: int(band[1:]))
wavelengths_nm = [int(band[1:]) for band in ordered_bands]
box_width = 0.8 * min(
    right - left for left, right in zip(wavelengths_nm, wavelengths_nm[1:])
)

for group_number, (_, group) in enumerate(GROUPS):
    fig.add_trace(
        go.Box(
            x=wavelengths_nm * len(group),
            y=group[ordered_bands].to_numpy().ravel(),
            width=box_width,
            boxpoints=False,
            hovertemplate="Wavelength=%{x} nm<br>Reflectance=%{y}<extra></extra>",
        ),
        row=group_number // n_cols + 1,
        col=group_number % n_cols + 1,
    )

fig.update_xaxes(
    type="linear",
    title_text="Wavelength [nm]",
    tickmode="array",
    tickvals=wavelengths_nm[::10],
)
fig.update_yaxes(title_text="Reflectance [%]")
fig.show()